# 🌊 OceanXRay — Full Pipeline on Google Colab

**What this notebook does (top to bottom):**

| Step | What |
|------|------|
| 0 | Mount Google Drive + set project root |
| 1 | Install all Python dependencies |
| 2 | Clone / upload OceanXRay source code |
| 3 | Authenticate with Copernicus Marine + ECMWF CDS |
| 4 | Download 6 real-world datasets → Google Drive |
| 5 | Run Day 1 pipeline (preprocess → ML-ready dataset) |
| 6 | Run Day 2 pipeline (Climatology + MLP) |
| 7 | Run Day 3 pipeline (CNN training + evaluation) |
| 8 | Run Stage 5 (Ablation + Regional + Embedding) |
| 9 | Display key results + plots |

> ⚠️ **Requirements**: Colab Pro + Google Drive with ≥ 40 GB free space.  
> All results and model checkpoints are saved to Drive and survive session restarts.


## Step 0 — Mount Google Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

# ── USER CONFIG ──────────────────────────────────────────────────────────────
# Change this if you want the project in a different Drive folder.
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/oceanxray')
# ─────────────────────────────────────────────────────────────────────────────

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project root: {DRIVE_PROJECT_ROOT}")

# Symlink to /content/oceanxray for shorter paths in subprocess calls
COLAB_ROOT = Path('/content/oceanxray')
if not COLAB_ROOT.exists():
    os.symlink(str(DRIVE_PROJECT_ROOT), str(COLAB_ROOT))

# Add project root to Python path so imports work
if str(COLAB_ROOT) not in sys.path:
    sys.path.insert(0, str(COLAB_ROOT))

os.chdir(COLAB_ROOT)
print(f"Working directory: {os.getcwd()}")


## Step 1 — Install dependencies

> Takes ~2 min on first run. Already-installed packages are skipped.

In [ ]:
%%capture install_log
!pip install -q \
    xarray>=2023.1.0 \
    numpy>=1.23 \
    pandas>=1.5 \
    netCDF4>=1.6 \
    scipy>=1.9 \
    matplotlib>=3.6 \
    dask>=2023.1.0 \
    torch>=2.0 \
    scikit-learn>=1.0 \
    copernicusmarine>=1.0.0 \
    cdsapi>=0.7.0

print("All packages installed.")
print(install_log.stdout[-500:] if len(install_log.stdout) > 500 else install_log.stdout)


## Step 2 — Upload OceanXRay source code

**Option A (recommended):** Upload the project zip to Drive and extract it here.  
**Option B:** Use the Colab file upload widget below.

Run whichever cell matches your situation, then skip the other.


In [ ]:
# ── OPTION A: Extract zip already in Drive ───────────────────────────────────
# Change ZIP_PATH to where you placed the zip in Drive.
ZIP_PATH = Path('/content/drive/MyDrive/oceanxray_with_real_data.zip')

if ZIP_PATH.exists():
    import zipfile
    print(f"Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DRIVE_PROJECT_ROOT)
    print("Extraction complete.")
else:
    print(f"Zip not found at {ZIP_PATH}. Use Option B or place the zip there.")


In [ ]:
# ── OPTION B: Upload zip via widget ─────────────────────────────────────────
from google.colab import files
import zipfile, io

uploaded = files.upload()   # triggers upload dialog

for filename, data in uploaded.items():
    print(f"Extracting {filename} ({len(data)/1e6:.1f} MB) ...")
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        z.extractall(DRIVE_PROJECT_ROOT)
    print("Done.")


In [ ]:
# Verify code structure
import os, sys
from pathlib import Path

COLAB_ROOT = Path('/content/oceanxray')
os.chdir(COLAB_ROOT)
if str(COLAB_ROOT) not in sys.path:
    sys.path.insert(0, str(COLAB_ROOT))

required = ['config.py', 'run_pipeline.py', 'run_day2.py', 'run_day3.py',
            'src/loaders.py', 'src/cnn_model.py']
missing = [f for f in required if not (COLAB_ROOT / f).exists()]
if missing:
    print("MISSING files:", missing)
    print("Re-check your zip / Drive path.")
else:
    print("All required source files found.")
    !ls -la


## Step 3 — Authentication

### 3a. Copernicus Marine Service
Register free at https://data.marine.copernicus.eu if you haven't already.


In [ ]:
# Copernicus Marine login (interactive — enter username + password when prompted)
import copernicusmarine
copernicusmarine.login()


### 3b. ECMWF CDS (ERA5 winds)
1. Register at https://cds.climate.copernicus.eu
2. Get your API key from https://cds.climate.copernicus.eu/profile
3. Paste UID and API key in the cell below.


In [ ]:
import os
from pathlib import Path

# ── FILL IN YOUR CDS CREDENTIALS ────────────────────────────────────────────
CDS_UID     = "YOUR_UID_HERE"          # e.g. "123456"
CDS_API_KEY = "YOUR_API_KEY_HERE"      # e.g. "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
# ─────────────────────────────────────────────────────────────────────────────

cdsapirc = Path.home() / ".cdsapirc"
cdsapirc.write_text(f"url: https://cds.climate.copernicus.eu/api\nkey: {CDS_UID}:{CDS_API_KEY}\n")
print(f"CDS config written to {cdsapirc}")

# Quick test
import cdsapi
try:
    client = cdsapi.Client(quiet=True)
    print("CDS authentication OK.")
except Exception as e:
    print(f"CDS auth failed: {e}")
    print("Double-check UID and API key above.")


## Step 4 — Download real-world datasets → Drive

Each dataset is downloaded once and skipped on re-runs.  
Estimated total: **18–30 GB**. GLORYS thetao alone may take **30–60 min**.

Run the cells in order; you can skip any dataset already on Drive.


In [ ]:
# Ensure download dirs exist
from pathlib import Path
RAW = Path('/content/oceanxray/data/raw')
for d in ['sst', 'ssh', 'currents', 'winds', 'glorys', 'argo']:
    (RAW / d).mkdir(parents=True, exist_ok=True)
print("Data directories ready.")


In [ ]:
# ── 4a. SST — NOAA OISST v2.1 (~300-400 MB, no auth needed) ─────────────────
import urllib.request

out_file = RAW / 'sst' / 'oisst_v21_daily_2021_2024.nc'
if out_file.exists():
    print(f"Already exists: {out_file.name} ({out_file.stat().st_size/1e6:.0f} MB) — skipping.")
else:
    base = "https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180.nc"
    params = ("?analysed_sst"
              "[(2021-01-01T12:00:00Z):1:(2024-12-31T12:00:00Z)]"
              "[(5.0):1:(30.0)]"
              "[(45.0):1:(105.0)]")
    url = base + params
    print(f"Downloading OISST SST...")
    print(f"URL: {url}")
    urllib.request.urlretrieve(url, str(out_file))
    print(f"Done: {out_file.stat().st_size/1e6:.0f} MB")


In [ ]:
# ── 4b. SSH / SLA — Copernicus SEALEVEL 0.125° (~1-2 GB) ────────────────────
import copernicusmarine

out_file = RAW / 'ssh' / 'ssh_sealevel_my_2021_2024.nc'
if out_file.exists():
    print(f"Already exists: {out_file.name} ({out_file.stat().st_size/1e6:.0f} MB) — skipping.")
else:
    print("Downloading Copernicus SSH/SLA ...")
    copernicusmarine.subset(
        dataset_id="cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D",
        variables=["sla", "adt"],
        minimum_latitude=5.0, maximum_latitude=30.0,
        minimum_longitude=45.0, maximum_longitude=105.0,
        start_datetime="2021-01-01", end_datetime="2024-12-31",
        output_filename=str(out_file),
        force_download=True,
    )
    print(f"Done: {out_file.stat().st_size/1e6:.0f} MB")


In [ ]:
# ── 4c. Surface Currents — GLORYS uo, vo (surface only, ~1-2 GB) ─────────────
out_file = RAW / 'currents' / 'glorys_currents_surface_2021_2024.nc'
if out_file.exists():
    print(f"Already exists: {out_file.name} ({out_file.stat().st_size/1e6:.0f} MB) — skipping.")
else:
    print("Downloading GLORYS surface currents (uo, vo) ...")
    copernicusmarine.subset(
        dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
        variables=["uo", "vo"],
        minimum_latitude=5.0, maximum_latitude=30.0,
        minimum_longitude=45.0, maximum_longitude=105.0,
        start_datetime="2021-01-01", end_datetime="2024-12-31",
        minimum_depth=0.0, maximum_depth=1.0,
        output_filename=str(out_file),
        force_download=True,
    )
    print(f"Done: {out_file.stat().st_size/1e6:.0f} MB")


In [ ]:
# ── 4d. GLORYS thetao 0-1100 m (~15-25 GB) ───────────────────────────────────
# LARGEST DOWNLOAD — may take 30-60 min. Do NOT interrupt once started.
# If it fails midway, re-run this cell; the server supports re-requests
# and the output file will be overwritten cleanly.

out_file = RAW / 'glorys' / 'glorys_thetao_0_1100m_2021_2024.nc'
if out_file.exists():
    print(f"Already exists: {out_file.name} ({out_file.stat().st_size/1e6:.0f} MB) — skipping.")
else:
    print("Downloading GLORYS thetao (0-1100 m). This will take a while...")
    copernicusmarine.subset(
        dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
        variables=["thetao"],
        minimum_latitude=5.0, maximum_latitude=30.0,
        minimum_longitude=45.0, maximum_longitude=105.0,
        start_datetime="2021-01-01", end_datetime="2024-12-31",
        minimum_depth=0.0, maximum_depth=1100.0,
        output_filename=str(out_file),
        force_download=True,
    )
    print(f"Done: {out_file.stat().st_size/1e6:.0f} MB")


In [ ]:
# ── 4e. ERA5 Daily Winds (u10, v10) — ~600 MB-1 GB total ────────────────────
# One CDS request per year (queue limit). ~5-15 min per year.
import cdsapi
client = cdsapi.Client()

for year in ["2021", "2022", "2023", "2024"]:
    out_file = RAW / 'winds' / f'era5_daily_winds_{year}.nc'
    if out_file.exists():
        print(f"Already exists: era5_daily_winds_{year}.nc ({out_file.stat().st_size/1e6:.0f} MB) — skipping.")
        continue
    print(f"Requesting ERA5 daily winds for {year} ...")
    months = [f"{m:02d}" for m in range(1, 13)]
    days   = [f"{d:02d}" for d in range(1, 32)]
    client.retrieve(
        "derived-era5-single-levels-daily-statistics",
        {
            "product_type": "reanalysis",
            "variable": ["10m_u_component_of_wind", "10m_v_component_of_wind"],
            "year": year, "month": months, "day": days,
            "daily_statistic": "daily_mean",
            "time_zone": "utc+0:00", "frequency": "1_hourly",
            "area": [30, 45, 5, 105],   # N W S E
            "format": "netcdf",
        },
        str(out_file),
    )
    print(f"Done: {out_file.stat().st_size/1e6:.0f} MB")


In [ ]:
# ── Verify all datasets ───────────────────────────────────────────────────────
checks = [
    (RAW / 'sst'      / 'oisst_v21_daily_2021_2024.nc',         'SST'),
    (RAW / 'ssh'      / 'ssh_sealevel_my_2021_2024.nc',         'SSH'),
    (RAW / 'currents' / 'glorys_currents_surface_2021_2024.nc', 'Currents'),
    (RAW / 'glorys'   / 'glorys_thetao_0_1100m_2021_2024.nc',   'GLORYS thetao'),
]

all_ok = True
for path, label in checks:
    if path.exists():
        mb = path.stat().st_size / 1e6
        print(f"  OK  {label:20s}  {path.name}  ({mb:.0f} MB)")
    else:
        print(f"  XX  {label:20s}  NOT FOUND")
        all_ok = False

wind_files = sorted((RAW / 'winds').glob('era5_daily_winds_*.nc'))
if wind_files:
    total_mb = sum(f.stat().st_size for f in wind_files) / 1e6
    print(f"  OK  {'Winds':20s}  {len(wind_files)} file(s)  ({total_mb:.0f} MB total)")
else:
    print(f"  XX  {'Winds':20s}  NO FILES FOUND")
    all_ok = False

print()
print("ALL DATASETS READY." if all_ok else "SOME DATASETS MISSING — re-run failed cells above.")


## Step 5 — Day 1: Preprocessing → ML-ready dataset

Runs the full Day 1 pipeline:
`load → inspect → crop → temporal align → regrid → vertical interp → mask → split → normalize → save`

Output: `data/processed/ml_ready_dataset.nc` + normalization stats + metadata.


In [ ]:
import os, sys
from pathlib import Path

COLAB_ROOT = Path('/content/oceanxray')
os.chdir(COLAB_ROOT)
if str(COLAB_ROOT) not in sys.path:
    sys.path.insert(0, str(COLAB_ROOT))

# Check if already done (skip if ml_ready_dataset.nc exists)
ml_ready = COLAB_ROOT / 'data' / 'processed' / 'ml_ready_dataset.nc'
if ml_ready.exists():
    print(f"Day 1 output already exists ({ml_ready.stat().st_size/1e6:.0f} MB). Skipping.")
    print("Delete data/processed/ml_ready_dataset.nc to force a re-run.")
else:
    print("Running Day 1 pipeline...")
    !python run_pipeline.py
    if ml_ready.exists():
        print(f"\nDay 1 complete. ML-ready dataset: {ml_ready.stat().st_size/1e6:.0f} MB")
    else:
        raise RuntimeError("Day 1 FAILED — ml_ready_dataset.nc not created. Check output above.")


## Step 6 — Day 2: Climatology + Point-MLP

Gate: MLP must beat climatology on the **validation** split before CNN is started.


In [ ]:
mlp_checkpoint = COLAB_ROOT / 'models' / 'mlp_best.pt'
if mlp_checkpoint.exists():
    print(f"Day 2 checkpoint already exists. Skipping training.")
    print("Delete models/mlp_best.pt to force a re-run.")
else:
    print("Running Day 2 pipeline (Climatology + MLP training)...")
    !python run_day2.py
    if mlp_checkpoint.exists():
        print("\nDay 2 complete. MLP checkpoint saved.")
    else:
        raise RuntimeError("Day 2 FAILED. Check output above.")


## Step 7 — Day 3: CNN Training + 3-Way Evaluation

Gate: CNN must beat MLP on the **validation** split.  
Also extracts CNN embeddings → `results/embeddings.npz`.


In [ ]:
cnn_checkpoint = COLAB_ROOT / 'models' / 'cnn_best.pt'
if cnn_checkpoint.exists():
    print(f"Day 3 checkpoint already exists. Skipping training.")
    print("Delete models/cnn_best.pt to force a re-run.")
else:
    print("Running Day 3 pipeline (CNN training)...")
    # Use GPU if available
    import torch
    device_info = "GPU" if torch.cuda.is_available() else "CPU"
    print(f"Training on: {device_info}")
    !python run_day3.py
    if cnn_checkpoint.exists():
        print("\nDay 3 complete. CNN checkpoint saved.")
    else:
        raise RuntimeError("Day 3 FAILED. Check output above.")


## Step 8 — Stage 5: Ablation + Regional Analysis + Embedding

- **Ablation**: 1×1 center-MLP vs 3×3 CNN — does spatial context help?
- **Regional**: Arabian Sea vs Bay of Bengal CNN performance
- **Embedding**: PCA / t-SNE of CNN embeddings coloured by region/season/SST


In [ ]:
print("Running Stage 5 ...")
!python run_stage5.py


## Step 9 — Results & Plots

In [ ]:
import json
from pathlib import Path

COLAB_ROOT = Path('/content/oceanxray')

# ── 9a. Ablation table ──────────────────────────────────────────────────────
import pandas as pd

ablation_csv = COLAB_ROOT / 'results' / 'ablation' / 'ablation_table.csv'
if ablation_csv.exists():
    print("=== ABLATION: 1x1 Center-MLP vs 3x3 CNN ===")
    df = pd.read_csv(ablation_csv)
    print(df.to_string(index=False))
else:
    print("Ablation table not found. Run Stage 5 first.")


In [ ]:
# ── 9b. Regional metrics (GLORYS) ──────────────────────────────────────────
regional_json = COLAB_ROOT / 'results' / 'regional' / 'glorys_regional_metrics.json'
if regional_json.exists():
    with open(regional_json) as f:
        reg = json.load(f)
    print("=== REGIONAL METRICS (CNN vs GLORYS, test split) ===")
    for region, metrics in reg.items():
        if metrics.get('skipped', True):
            print(f"  {region}: skipped (0 samples)")
        else:
            ov = metrics['overall']
            print(f"  {region} (n={metrics['n_samples']}): "
                  f"RMSE={ov['rmse']:.4f}°C, MAE={ov['mae']:.4f}°C, "
                  f"Bias={ov['bias']:.4f}°C, Corr={ov['correlation']:.4f}")
else:
    print("Regional metrics not found. Run Stage 5 first.")


In [ ]:
# ── 9c. Show all result plots ───────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

plot_dirs = [
    COLAB_ROOT / 'results' / 'plots',
    COLAB_ROOT / 'results' / 'ablation' / 'plots',
    COLAB_ROOT / 'results' / 'regional' / 'plots',
    COLAB_ROOT / 'results' / 'embedding' / 'plots',
]

all_plots = []
for d in plot_dirs:
    if d.exists():
        all_plots.extend(sorted(d.glob('*.png')))

if not all_plots:
    print("No plots found yet. Run Stages 5-8 first.")
else:
    print(f"Found {len(all_plots)} plots.")
    for plot_path in all_plots:
        fig, ax = plt.subplots(figsize=(12, 6))
        img = mpimg.imread(str(plot_path))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(plot_path.stem, fontsize=10)
        plt.tight_layout()
        plt.show()


In [ ]:
# ── 9d. Embedding PCA variance summary ─────────────────────────────────────
ev_json = COLAB_ROOT / 'results' / 'embedding' / 'explained_variance.json'
if ev_json.exists():
    with open(ev_json) as f:
        ev = json.load(f)
    print("=== CNN EMBEDDING PCA ===")
    ratio = ev['explained_variance_ratio']
    print(f"PC1: {ratio[0]*100:.1f}%   PC2: {ratio[1]*100:.1f}%   "
          f"Total (2 PCs): {ev['cumulative_variance_2pc']*100:.1f}%")
    print(f"Points: {ev['n_points_used_for_pca']} / {ev['n_points_total']}")


In [ ]:
# ── 9e. Disk usage summary ──────────────────────────────────────────────────
import shutil
from pathlib import Path

dirs_to_check = {
    'Raw data':       COLAB_ROOT / 'data' / 'raw',
    'Processed data': COLAB_ROOT / 'data' / 'processed',
    'Models':         COLAB_ROOT / 'models',
    'Results':        COLAB_ROOT / 'results',
}

print("=== DISK USAGE ===")
total = 0
for label, d in dirs_to_check.items():
    if d.exists():
        size = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 1e9
        total += size
        print(f"  {label:20s}  {size:.2f} GB")
    else:
        print(f"  {label:20s}  (not found)")
print(f"  {'TOTAL':20s}  {total:.2f} GB")
